# v05 — Code Execution Pipeline

End-to-end Colab runner for `v05_code_execution`. Three-stage pipeline:
1. **Stage 1 (Router)**: Classify each question into (domain, answer_type) — batched
2. **Stage 2 (Code Gen)**: Generate Python code to compute the answer — per question
3. **Stage 3 (Execute)**: Run the code, parse output, format answer — per question

Falls back to v03-style direct solve for `text_only`/`mixed` answer types, or when code generation/execution fails.

**v05+ only evaluates on golden data** (DeepSeek-rewritten CoT), not on `full_test.csv`.

All knobs (model, dtype, batch size, max tokens, ...) live in [`app/physics_solution/config.py`](../../config.py) — edit there, not here.

## 1. Mount Drive + chdir

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_ROOT = '/content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils'  # change if needed
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

In [ ]:
!pip -q install -r app/physics_solution/requirements.txt

### Install Qwen3.5 fast-attention kernels (`flash-linear-attention` + `causal-conv1d`)

On Colab Oct-2025+ runtimes both wheels install cleanly. If `causal_conv1d: False` shows up, the model still runs via torch fallback at ~3× slower — accuracy is unaffected.

In [ ]:
from app.physics_solution.shared.colab.setup import install_fast_kernels
install_fast_kernels()

In [ ]:
from app.physics_solution.shared.colab.setup import setup_colab
setup_colab(repo_root=REPO_ROOT)

In [ ]:
!nvidia-smi -L

## 2. Test set path

v05+ only evaluates on **golden data** (DeepSeek-rewritten CoT).

| File | Rows | Scope |
|---|---|---|
| `deepseek-v4-pro_golden_data.csv` | 1352 | All answer types — golden CoT |

In [ ]:
import os

GOLDEN_TEST_FILE = 'app/physics_solution/data/golden/deepseek-v4-pro_golden_data.csv'
GOLDEN_OUT_FILE  = 'app/physics_solution/versions/v05_code_execution/output/results_golden.json'

assert os.path.exists(GOLDEN_TEST_FILE), f"Not found: {GOLDEN_TEST_FILE}"
print(f'Golden: {GOLDEN_TEST_FILE}')

## 3. Ensure few-shot examples exist

v05 uses v03's few-shot examples for the **fallback** path (direct solve when code execution fails).
The runner automatically falls back to v03's `examples.json` if v05 doesn't have its own.

In [ ]:
import os
v05_examples = 'app/physics_solution/versions/v05_code_execution/input/examples.json'
v03_examples = 'app/physics_solution/versions/v03_routed_fewshot/input/examples.json'

if os.path.exists(v05_examples):
    print('v05 examples exist:', v05_examples)
elif os.path.exists(v03_examples):
    print('v03 examples exist (v05 will use these as fallback):', v03_examples)
else:
    print('No examples found. Running v03 select_fewshot...')
    !python -m app.physics_solution.versions.v03_routed_fewshot.select_fewshot --k 3 --seed 42

## 4. Quick smoke test (5 questions)

Run a small sample first to verify the pipeline works before committing to the full 1352-question run.

**Note**: v05 is slower per-question than v03/v04 because code generation is sequential (not batched). Expect ~3-5s per question vs ~0.6s for v03.

In [ ]:
!python -m app.physics_solution.cli.inference \
    --version v05_code_execution \
    --test-file {GOLDEN_TEST_FILE} \
    --limit 5 \
    --batch-size 80

## 5. Full inference (golden data)

Three-stage pipeline: classify (batched) → code-gen + execute (per question, with retry) → fallback to direct solve.

Expected timing:
- Stage 1 (classify): ~50s (batched, same as v03/v04)
- Stages 2+3 (code gen + execute): ~3-5s per question × 1352 = ~70-110 min
- Total: ~75-115 min (vs ~15 min for v03)

In [ ]:
!python -m app.physics_solution.cli.inference \
    --version v05_code_execution \
    --test-file {GOLDEN_TEST_FILE} \
    --out {GOLDEN_OUT_FILE} \
    --batch-size 80

## 6. Inspect results

In [ ]:
import json, pandas as pd

data = json.loads(open(GOLDEN_OUT_FILE, encoding='utf-8').read())
summary = data['summary']

print('='*60)
print('  v05 Code Execution — Golden Data Results')
print('='*60)
print(f"\nAccuracy: {summary['correct']}/{summary['n']} = {summary['accuracy']:.4f}")
print(f"Mean latency: {summary['mean_latency_s']:.2f} s/question")
print(f"\nSolve method breakdown:")
for method, count in summary.get('method_counts', {}).items():
    pct = count / summary['n'] * 100
    print(f"  {method:20s}: {count:5d} ({pct:.1f}%)")

### 6a. Accuracy by solve method

In [ ]:
df = pd.DataFrame(data['rows'])
df['solve_method'] = df['extra'].apply(lambda x: x.get('solve_method', 'unknown') if isinstance(x, dict) else 'unknown')

print('Accuracy by solve method:')
for method, sub in df.groupby('solve_method'):
    correct = sub['is_correct'].sum()
    total = len(sub)
    print(f"  {method:20s}: {correct}/{total} = {correct/total:.3f}")

print(f"\n{'Overall':20s}: {df['is_correct'].sum()}/{len(df)} = {df['is_correct'].mean():.3f}")

### 6b. Per-domain accuracy

In [ ]:
import re
PREFIX_RE = re.compile(r"^([A-Z]+)")
df['true_domain'] = df['id'].apply(lambda x: PREFIX_RE.match(str(x)).group(1) if PREFIX_RE.match(str(x)) else '')
df['routed_domain'] = df['extra'].apply(lambda x: x.get('routed_domain', '') if isinstance(x, dict) else '')

print('Per-domain accuracy:')
for domain, sub in df.groupby('true_domain'):
    correct = sub['is_correct'].sum()
    total = len(sub)
    # Also show code execution rate for this domain
    code_exec = (sub['solve_method'] == 'code_execution').sum()
    fallback = (sub['solve_method'] == 'fallback').sum()
    direct = (sub['solve_method'] == 'direct_solve').sum()
    print(f"  {domain:6s}: {correct:3d}/{total:3d} = {correct/total:.3f}  "
          f"(code:{code_exec} fallback:{fallback} direct:{direct})")

### 6c. Code execution failure analysis

In [ ]:
# Analyze why code execution failed (fallback cases)
fallback_rows = df[df['solve_method'] == 'fallback'].copy()
print(f"Total fallback cases: {len(fallback_rows)}")

if len(fallback_rows) > 0:
    fallback_rows['stderr'] = fallback_rows['extra'].apply(
        lambda x: x.get('execution_stderr', '') if isinstance(x, dict) else ''
    )
    fallback_rows['retry_count'] = fallback_rows['extra'].apply(
        lambda x: x.get('retry_count', 0) if isinstance(x, dict) else 0
    )
    fallback_rows['had_code'] = fallback_rows['extra'].apply(
        lambda x: x.get('generated_code') is not None if isinstance(x, dict) else False
    )
    
    print(f"\n  No code block extracted: {(~fallback_rows['had_code']).sum()}")
    print(f"  Had code but execution failed: {fallback_rows['had_code'].sum()}")
    print(f"  Retried at least once: {(fallback_rows['retry_count'] > 0).sum()}")
    
    # Show top error types
    print(f"\nTop error messages (first 50 chars):")
    error_counts = fallback_rows['stderr'].apply(lambda x: x[:50] if x else '(empty)').value_counts().head(10)
    for err, count in error_counts.items():
        print(f"  [{count:3d}] {err}")

### 6d. Wrong predictions breakdown

In [ ]:
wrong = df[~df['is_correct']].copy()
wrong['reached_final'] = wrong['completion'].str.contains('FINAL ANSWER', case=False, regex=False)
print(f"Wrong: {len(wrong)} / {len(df)}")
print(f"  ... never reached FINAL ANSWER: {(~wrong['reached_final']).sum()}")

print(f"\nWrong by solve method:")
for method, sub in wrong.groupby('solve_method'):
    print(f"  {method:20s}: {len(sub)}")

display(wrong[['id', 'gold_answer', 'pred_numeric', 'pred_unit', 'gold_unit', 'solve_method', 'reached_final']].head(20))

## 7. Routing accuracy

In [ ]:
df['domain_correct'] = df['routed_domain'] == df['true_domain']
print(f"Domain routing accuracy: {df['domain_correct'].sum()}/{len(df)} = {df['domain_correct'].mean():.3f}")

print("\nPer-domain routing accuracy:")
for domain, sub in df.groupby('true_domain'):
    acc = sub['domain_correct'].mean()
    print(f"  {domain}: {sub['domain_correct'].sum()}/{len(sub)} = {acc:.3f}")

misrouted = df[~df['domain_correct']]
if len(misrouted) > 0:
    print(f"\nMisrouted examples ({len(misrouted)} total):")
    print(pd.crosstab(misrouted['true_domain'], misrouted['routed_domain'], margins=True))

## 8. Comparison with previous versions

Compare v05 code execution against v03 (routed few-shot) and v04 (optimized routing) golden results.

In [ ]:
import os

prev_versions = {
    'v03_golden': 'app/physics_solution/versions/v03_routed_fewshot/output/results_golden.json',
    'v04_golden': 'app/physics_solution/versions/v04_optimized_routing/output/results_golden.json',
}

print('=== Version Comparison (golden data) ===')
print(f"  {'Version':<15s} {'Accuracy':>10s} {'Correct':>10s} {'Latency':>10s}")
print(f"  {'-'*15} {'-'*10} {'-'*10} {'-'*10}")

for label, path in prev_versions.items():
    if os.path.exists(path):
        prev = json.loads(open(path, encoding='utf-8').read())['summary']
        print(f"  {label:<15s} {prev['accuracy']:>10.4f} {prev['correct']:>5d}/{prev['n']:<4d} {prev['mean_latency_s']:>8.2f} s")
    else:
        print(f"  {label:<15s} {'(not found)':>10s}")

# v05
print(f"  {'v05_golden':<15s} {summary['accuracy']:>10.4f} {summary['correct']:>5d}/{summary['n']:<4d} {summary['mean_latency_s']:>8.2f} s")

# Delta from best previous
best_prev_acc = 0
for path in prev_versions.values():
    if os.path.exists(path):
        prev = json.loads(open(path, encoding='utf-8').read())['summary']
        best_prev_acc = max(best_prev_acc, prev['accuracy'])
if best_prev_acc > 0:
    delta = summary['accuracy'] - best_prev_acc
    print(f"\n  Delta from best previous: {delta:+.4f} ({delta*100:+.2f} pp)")

### 8a. Per-question diff: what did code execution fix/break?

Compare v05 vs v03 on a per-question basis to see which questions were fixed by code execution and which regressed.

In [ ]:
v03_golden_path = 'app/physics_solution/versions/v03_routed_fewshot/output/results_golden.json'

if os.path.exists(v03_golden_path):
    v03_data = json.loads(open(v03_golden_path, encoding='utf-8').read())
    v03_df = pd.DataFrame(v03_data['rows'])
    
    # Merge on question ID
    merged = df[['id', 'is_correct', 'solve_method']].merge(
        v03_df[['id', 'is_correct']],
        on='id', suffixes=('_v05', '_v03')
    )
    
    fixed = merged[(merged['is_correct_v05']) & (~merged['is_correct_v03'])]
    broken = merged[(~merged['is_correct_v05']) & (merged['is_correct_v03'])]
    both_right = merged[(merged['is_correct_v05']) & (merged['is_correct_v03'])]
    both_wrong = merged[(~merged['is_correct_v05']) & (~merged['is_correct_v03'])]
    
    print(f'Questions: {len(merged)}')
    print(f'  Both correct:    {len(both_right)}')
    print(f'  Both wrong:      {len(both_wrong)}')
    print(f'  Fixed by v05:    {len(fixed)}  (v03 wrong -> v05 correct)')
    print(f'  Broken by v05:   {len(broken)}  (v03 correct -> v05 wrong)')
    print(f'  Net improvement: {len(fixed) - len(broken):+d}')
    
    if len(fixed) > 0:
        print(f'\nFixed questions by solve method:')
        print(fixed['solve_method'].value_counts().to_string())
    
    if len(broken) > 0:
        print(f'\nBroken questions by solve method:')
        print(broken['solve_method'].value_counts().to_string())
        print(f'\nFirst 10 broken question IDs: {broken["id"].head(10).tolist()}')
else:
    print('v03 golden results not found — run v03 first for comparison.')

## 9. Sample generated code inspection

Look at a few examples of generated code to verify quality.

In [ ]:
code_rows = df[df['solve_method'] == 'code_execution'].copy()

# Show 3 correct and 3 wrong examples
for label, subset in [('CORRECT', code_rows[code_rows['is_correct']].head(3)),
                       ('WRONG', code_rows[~code_rows['is_correct']].head(3))]:
    print(f"\n{'='*70}")
    print(f"  {label} code execution examples")
    print(f"{'='*70}")
    for _, row in subset.iterrows():
        extra = row['extra'] if isinstance(row['extra'], dict) else {}
        code = extra.get('generated_code', '(no code)')
        print(f"\n--- {row['id']} ---")
        print(f"Q: {row['question'][:120]}...")
        print(f"Gold: {row['gold_answer']} {row['gold_unit']}")
        print(f"Pred: {row['pred_numeric']} {row['pred_unit']}")
        print(f"Code:\n{code}")
        print()

## 10. Push to HF (optional)

In [ ]:
# Uncomment to push:
# !python -m app.physics_solution.cli.upload_model \
#     --version-num 5 --strategy code_execution \
#     --results {GOLDEN_OUT_FILE} \
#     --test-file {GOLDEN_TEST_FILE}

## 11. Deep error analysis

Open [`app/physics_solution/eda/notebooks/error_analysis.ipynb`](../../eda/notebooks/error_analysis.ipynb) to categorise every wrong row into fail modes.